# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [4]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [8]:
# Desarrollo de la actividad 1
# Preparar dimension y mostrar la distribución por tipo.

divipola["codigo_municipio"] = (
    divipola["Código Municipio"]
    .astype("string")
    .str.strip()
)

# Construir la dimensión
dimension = divipola[[
    "codigo_municipio",
    "Nombre Departamento",
    "Nombre Municipio",
    "Tipo: Municipio / Isla / Área no municipalizada"
]].copy()

# Renombrar las columnas
dimension = dimension.rename(columns={
    "Nombre Departamento": "nombre_departamento",
    "Nombre Municipio": "nombre_municipio",
    "Tipo: Municipio / Isla / Área no municipalizada": "clasificacion_territorial"
})

print(dimension.head())

# Comprobar formato y unicidad del código.

codigo_valido = dimension["codigo_municipio"].str.fullmatch(
    r"[0-9]{5}",
    na=False
)

print("Códigos con cinco cifras:", codigo_valido.sum())
print("Códigos inválidos:", (~codigo_valido).sum())

# Verificar valores vacíos
print(
    "Códigos vacíos:",
    dimension["codigo_municipio"].isna().sum()
    + dimension["codigo_municipio"].eq("").sum()
)

# Verificar unicidad
print("¿El código es único?:", dimension["codigo_municipio"].is_unique)

# Verificar total de entidades
print("Total de entidades:", len(dimension))

# Conteo de entidades territoriales
conteo_tipos = dimension["clasificacion_territorial"].value_counts(
    dropna=False
)

print(conteo_tipos)

  codigo_municipio nombre_departamento nombre_municipio  \
0            05001           ANTIOQUIA         MEDELLÍN   
1            05002           ANTIOQUIA        ABEJORRAL   
2            05004           ANTIOQUIA         ABRIAQUÍ   
3            05021           ANTIOQUIA       ALEJANDRÍA   
4            05030           ANTIOQUIA            AMAGÁ   

  clasificacion_territorial  
0                 Municipio  
1                 Municipio  
2                 Municipio  
3                 Municipio  
4                 Municipio  
Códigos con cinco cifras: 1122
Códigos inválidos: 0
Códigos vacíos: 0
¿El código es único?: True
Total de entidades: 1122
clasificacion_territorial
Municipio                 1103
Área no municipalizada      18
Isla                         1
Name: count, dtype: int64


**Interpretación del resultado:**

La información de DIVIPOLA registra 1.122 entidades territoriales, de las cuales 1.103 corresponden a municipios, 18 a áreas no municipalizadas y 1 a isla. Los municipios representan la mayoría de las entidades incluidas en el archivo.

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [10]:
# Desarrollo de la actividad 2
# Preparar código y año en eva.
eva["codigo_municipio"] = (
    eva["Código Dane municipio"]
    .astype("string")
    .str.strip()
)

eva["anio"] = (
    eva["Año"]
    .astype("string")
    .str.strip()
)

# Identificar códigos compuestos únicamente por dígitos
es_numerico = eva["codigo_municipio"].str.fullmatch(
    r"[0-9]+",
    na=False
)

# Solo son válidos los códigos numéricos de 1 a 5 cifras
codigo_valido = (
    es_numerico
    & eva["codigo_municipio"].str.len().between(1, 5)
)

# Completar con ceros únicamente los códigos válidos
eva["codigo_municipio"] = (
    eva["codigo_municipio"]
    .where(codigo_valido)
    .str.zfill(5)
)

# Identificar códigos que requieren revisión
eva["codigo_revisar"] = ~codigo_valido

print("Total de registros EVA:", len(eva))
print("Códigos que requieren revisión:", eva["codigo_revisar"].sum())

combinaciones = eva[
    [
        "Código Dane municipio",
        "codigo_municipio",
        "Año",
        "anio",
        "codigo_revisar"
    ]
].drop_duplicates()

print("Combinaciones distintas de código y año:", len(combinaciones))

# Obtener sin_correspondencia con código municipal y año.
# Comprobar que DIVIPOLA tenga códigos únicos
assert dimension["codigo_municipio"].is_unique

# Comparar los códigos de EVA contra los de DIVIPOLA
comparacion = combinaciones.merge(
    dimension[["codigo_municipio"]],
    on="codigo_municipio",
    how="left",
    indicator=True,
    validate="many_to_one"
)

# Conservar códigos sin correspondencia o que requieren revisión
sin_correspondencia = comparacion.loc[
    (comparacion["_merge"] == "left_only")
    | comparacion["codigo_revisar"]
].copy()

# Clasificar el motivo
sin_correspondencia["motivo"] = "Código no encontrado en DIVIPOLA"

sin_correspondencia.loc[
    sin_correspondencia["codigo_revisar"],
    "motivo"
] = "Código vacío o mal formado"

# Mostrar el resultado, incluso si está vacío
print("Total de combinaciones revisadas:", len(combinaciones))
print("Total sin correspondencia:", len(sin_correspondencia))
print("\nListado de códigos sin correspondencia:")
print(sin_correspondencia.to_string(index=False))

Total de registros EVA: 166732
Códigos que requieren revisión: 0
Combinaciones distintas de código y año: 7691
Total de combinaciones revisadas: 7691
Total sin correspondencia: 0

Listado de códigos sin correspondencia:
Empty DataFrame
Columns: [Código Dane municipio, codigo_municipio, Año, anio, codigo_revisar, _merge, motivo]
Index: []


**Interpretación del resultado:**

El conjunto de datos EVA contiene 166.732 registros, de los cuales se obtuvieron 7.691 combinaciones distintas de código municipal y año. Durante la validación, no se encontraron códigos vacíos ni mal formados que requirieran revisión.
Al comparar estas combinaciones con la dimensión territorial mediante el código municipal, se obtuvo un total de 0 combinaciones sin correspondencia. Esto indica que todos los códigos municipales normalizados de EVA están presentes en la dimensión territorial utilizada como referencia.

## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [12]:
# Desarrollo de la actividad 3
# Crear papa_2024 y apta.
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"], errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"], errors="coerce")

papa_2024 = eva.loc[
    eva["Cultivo"].eq("Papa") & eva["anio"].eq("2024")
].copy()

condicion_apta = (
    papa_2024["area_ha"].gt(0) &
    papa_2024["produccion_t"].ge(0)
)

apta = papa_2024.loc[condicion_apta].copy()

# Mostrar las exclusiones del indicador y conciliar los conteos.
filas_papa_2024 = len(papa_2024)
filas_aptas = len(apta)
filas_excluidas = filas_papa_2024 - filas_aptas

print("Total de filas de papa en 2024:", filas_papa_2024)
print("Filas aptas para el indicador:", filas_aptas)
print("Filas excluidas del indicador:", filas_excluidas)

print("¿Se cumple la comprobación?",
      filas_papa_2024 == filas_aptas + filas_excluidas)

Total de filas de papa en 2024: 831
Filas aptas para el indicador: 779
Filas excluidas del indicador: 52
¿Se cumple la comprobación? True


**Interpretación del resultado:**

El proceso permitió identificar los registros del cultivo de papa correspondientes al año 2024 y separar aquellos que cumplen las condiciones necesarias para calcular el indicador de producción agrícola.
Las filas aptas son aquellas que tienen un área cosechada mayor que cero y una producción mayor o igual que cero. Por el contrario, se excluyen los registros que presentan un área igual a cero o negativa, una producción negativa o alguna de estas medidas ausente o inválida.

## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [14]:
# Desarrollo de la actividad 4
# Crear rendimiento con los agregados y rendimiento_t_ha.
rendimiento = (
    apta.groupby(
        [
            "codigo_municipio",
            "Cultivo",
            "Estado físico del cultivo",
            "anio"
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        cantidad_registros=("area_ha", "size"),
        produccion_total_t=("produccion_t", "sum"),
        area_total_ha=("area_ha", "sum")
    )
)

rendimiento["rendimiento_t_ha"] = (
    rendimiento["produccion_total_t"] / rendimiento["area_total_ha"]
)

display(rendimiento)

# Comprobar denominadores y cantidad de registros agrupados.
print("Total de filas aptas:", len(apta))
print("Suma de registros agrupados:", rendimiento["cantidad_registros"].sum())

print(
    "¿Coinciden los conteos?",
    rendimiento["cantidad_registros"].sum() == len(apta)
)

print(
    "¿Todas las áreas totales son positivas?",
    rendimiento["area_total_ha"].gt(0).all()
)

print(
    "¿El rendimiento está bien calculado?",
    (
        rendimiento["rendimiento_t_ha"]
        == rendimiento["produccion_total_t"] / rendimiento["area_total_ha"]
    ).all()
)

,codigo_municipio,Cultivo,Estado físico del cultivo,anio,cantidad_registros,produccion_total_t,area_total_ha,rendimiento_t_ha
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459
...,...,...,...,...,...,...,...,...
287,76834,Papa,En fresco,2024,2,8570.0,490.0,17.489796
288,85136,Papa,En fresco,2024,1,24.0,6.0,4.000000
289,86749,Papa,En fresco,2024,1,78.0,6.0,13.000000
290,86755,Papa,En fresco,2024,1,20.0,2.0,10.000000


Total de filas aptas: 779
Suma de registros agrupados: 779
¿Coinciden los conteos? True
¿Todas las áreas totales son positivas? True
¿El rendimiento está bien calculado? True


**Interpretación del resultado:**

El rendimiento se calcula como la razón entre las sumas de producción y área de cada grupo, no como el promedio simple de los rendimientos individuales. Así se obtiene un indicador ponderado por el área cosechada.

## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [15]:
# Desarrollo de la actividad 5
# Unir rendimiento con dimension.
rendimiento_municipal = rendimiento.merge(
    dimension,
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True
)

display(rendimiento_municipal)

# Verificar la conservación de filas y presentar la cobertura de la unión.
print("Filas antes de la unión:", len(rendimiento))
print("Filas después de la unión:", len(rendimiento_municipal))

print(
    "¿La cantidad de filas se conserva?",
    len(rendimiento) == len(rendimiento_municipal)
)

duplicados = dimension[
    dimension.duplicated("codigo_municipio", keep=False)
].sort_values("codigo_municipio")

display(duplicados)

sin_correspondencia = rendimiento_municipal.loc[
    rendimiento_municipal["_merge"] == "left_only"
].copy()

print("Filas sin correspondencia:", len(sin_correspondencia))

display(
    sin_correspondencia[
        [
            "codigo_municipio",
            "Cultivo",
            "Estado físico del cultivo",
            "anio"
        ]
    ].drop_duplicates()
)

,codigo_municipio,Cultivo,Estado físico del cultivo,anio,cantidad_registros,produccion_total_t,area_total_ha,rendimiento_t_ha,nombre_departamento,nombre_municipio,clasificacion_territorial,_merge
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000,ANTIOQUIA,MEDELLÍN,Municipio,both
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000,ANTIOQUIA,ABEJORRAL,Municipio,both
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000,ANTIOQUIA,ABRIAQUÍ,Municipio,both
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353,ANTIOQUIA,BARBOSA,Municipio,both
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459,ANTIOQUIA,BELMIRA,Municipio,both
...,...,...,...,...,...,...,...,...,...,...,...,...
287,76834,Papa,En fresco,2024,2,8570.0,490.0,17.489796,VALLE DEL CAUCA,TULUÁ,Municipio,both
288,85136,Papa,En fresco,2024,1,24.0,6.0,4.000000,CASANARE,LA SALINA,Municipio,both
289,86749,Papa,En fresco,2024,1,78.0,6.0,13.000000,PUTUMAYO,SIBUNDOY,Municipio,both
290,86755,Papa,En fresco,2024,1,20.0,2.0,10.000000,PUTUMAYO,SAN FRANCISCO,Municipio,both


Filas antes de la unión: 292
Filas después de la unión: 292
¿La cantidad de filas se conserva? True


,codigo_municipio,nombre_departamento,nombre_municipio,clasificacion_territorial


Filas sin correspondencia: 0


,codigo_municipio,Cultivo,Estado físico del cultivo,anio


**Interpretación del resultado:**

La cantidad de filas antes y después de la unión es idéntica y no existen registros left_only, todos los grupos de rendimiento quedaron relacionados con su información territorial sin perder registros ni multiplicarlos.

## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT / "rendimiento_municipal.csv"` y sin_correspondencia a `OUT / "codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [16]:
# Desarrollo de la actividad 6
# Exportar las dos tablas.
OUT.mkdir(parents=True, exist_ok=True)

rendimiento_municipal.to_csv(
    OUT / "rendimiento_municipal.csv",
    index=False
)

sin_correspondencia.to_csv(
    OUT / "codigos_sin_correspondencia.csv",
    index=False
)

print("Archivos exportados correctamente.")

archivo_rendimiento = OUT / "rendimiento_municipal.csv"
archivo_codigos = OUT / "codigos_sin_correspondencia.csv"

print("¿Existe rendimiento_municipal.csv?", archivo_rendimiento.exists())
print("¿Existe codigos_sin_correspondencia.csv?", archivo_codigos.exists())

print("Columnas del archivo de códigos:")
print(sin_correspondencia.columns.tolist())

print("Cantidad de códigos sin correspondencia:", len(sin_correspondencia))

# Explicar el denominador y la limitación temporal en la celda siguiente.

Archivos exportados correctamente.
¿Existe rendimiento_municipal.csv? True
¿Existe codigos_sin_correspondencia.csv? True
Columnas del archivo de códigos:
['codigo_municipio', 'Cultivo', 'Estado físico del cultivo', 'anio', 'cantidad_registros', 'produccion_total_t', 'area_total_ha', 'rendimiento_t_ha', 'nombre_departamento', 'nombre_municipio', 'clasificacion_territorial', '_merge']
Cantidad de códigos sin correspondencia: 0


**Interpretación del resultado:**

El rendimiento agrícola se calcula dividiendo la suma de la producción en toneladas entre la suma del área cosechada en hectáreas para cada grupo de municipio, cultivo, estado físico del cultivo y año. Se utiliza el área total como denominador porque los registros pueden representar superficies diferentes.

La dimensión territorial utilizada corresponde al catálogo con corte de 2024, mientras que los datos de EVA pueden pertenecer a años anteriores. Por tanto, una coincidencia del código municipal permite relacionar los registros con el catálogo actual, pero no garantiza que los límites o la organización territorial fueran iguales en el año de cada observación. Para realizar comparaciones históricas precisas, es necesario verificar los cambios de códigos, la creación de municipios y las modificaciones de límites territoriales.

## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.